# Milestone 8 - Faithfulness & citation guardrail

An **independent** verifier checks every sentence of a candidate answer before it reaches the user: (1) it must cite an existing source, (2) every number in it must appear in the cited source, and (3) an NLI cross-encoder (DeBERTa-v3-large trained on MNLI/FEVER/ANLI/LingNLI/WANLI) must judge it *entailed* by the cited source (max over sentence/table-row premises, p >= 0.5). Failures trigger one bounded regeneration with feedback, then removal of unverifiable sentences or an explicit "cannot be verified" fallback.

In [1]:
import os, sys, json, time, warnings, pathlib
warnings.filterwarnings("ignore")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.max_colwidth", 120)
from rag import config

In [2]:
from rag import guardrail, parsing
from rag.corpus import load_jsonl
from rag import evaluation
chunks = parsing.load_chunks()
src = [next({"doc": c.doc, "doc_type": c.doc_type, "page": c.page, "section": c.section, "text": c.parent_text}
            for c in chunks if c.doc.startswith("HX_Pump_Series") and c.page == 6)]
print(src[0]["text"])

## 6. HX-250 Centrifugal Pump

The HX-250 is recommended for boiler feed water applications. It uses a closed impeller and a mechanical shaft seal as standard.

| Parameter | Value |
|---|---|
| Maximum flow rate | 770 L/min |
| Maximum head | 133 m |
| Motor power | 111.9 kW |
| Impeller diameter | 152 mm |
| Dry weight | 159 kg |
| Inlet connection | DN80 |
| Maximum fluid temperature | 160 degC |
| Motor enclosure rating | IP65 |


### How the source is turned into NLI premises

In [3]:
for w in guardrail.premise_windows(src[0]): print("-", w)

- 6. HX-250 Centrifugal Pump: The HX-250 is recommended for boiler feed water applications. It uses a closed impeller and a mechanical shaft seal as standard. The Maximum flow rate is 770 L/min.
- 6. HX-250 Centrifugal Pump: The HX-250 is recommended for boiler feed water applications. It uses a closed impeller and a mechanical shaft seal as standard. The Maximum head is 133 m.
- 6. HX-250 Centrifugal Pump: The HX-250 is recommended for boiler feed water applications. It uses a closed impeller and a mechanical shaft seal as standard. The Motor power is 111.9 kW.
- 6. HX-250 Centrifugal Pump: The HX-250 is recommended for boiler feed water applications. It uses a closed impeller and a mechanical shaft seal as standard. The Impeller diameter is 152 mm.
- 6. HX-250 Centrifugal Pump: The HX-250 is recommended for boiler feed water applications. It uses a closed impeller and a mechanical shaft seal as standard. The Dry weight is 159 kg.
- 6. HX-250 Centrifugal Pump: The HX-250 is recommende

### Verifying a mixed answer

In [4]:
import re
model = re.search(r"HX-\d+", src[0]["section"]).group()
true_head = re.search(r"Maximum head \| (\d+ m)", src[0]["text"]).group(1)
answer = (f"The maximum head of the {model} is {true_head} [S1]. "
          f"The {model} has a dry weight of 12 kg [S1]. "
          f"The {model} is certified for use on offshore platforms [S1]. "
          f"It is the most popular pump in the range.")
v = guardrail.verify(answer, src, question=f"Tell me about the {model}")
print("status:", v["status"], "| faithfulness:", v["faithfulness"])
pd.DataFrame(v["claims"])[["claim", "citations", "numbers_ok", "entailment", "supported", "reason"]]

Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

status: partially_verified | faithfulness: 0.25


,claim,citations,numbers_ok,entailment,supported,reason
0,The maximum head of the HX-250 is 133 m.,[1],True,0.999512,True,
1,The HX-250 has a dry weight of 12 kg.,[1],False,0.001089,False,number not found in cited source
2,The HX-250 is certified for use on offshore platforms.,[1],True,0.000386,False,not entailed (p=0.00)
3,It is the most popular pump in the range.,[],False,0.000000,False,no citation


### Stress test on 120 held-out facts
For each fact: the correct reference sentence (should pass), the same sentence with the value swapped for another value from the same document (should be caught), and a true sentence about a different fact cited to the wrong page (should be caught).

In [5]:
test = load_jsonl(config.EVAL_DIR / "test_qa.jsonl"); facts = load_jsonl(config.EVAL_DIR / "facts.jsonl")
t = time.time()
st = evaluation.guardrail_stress(test, facts, chunks, n=120)
print(f"{st['n']} facts x 3 variants in {time.time() - t:.0f}s")
summary = pd.Series({k: st[k] for k in ("correct_accepted", "corrupted_caught", "foreign_caught")}).round(3)
(config.ARTIFACT_DIR / "guardrail_stress.json").write_text(json.dumps({k: v for k, v in st.items() if k != "examples"}, indent=2))
summary.to_frame("rate")

120 facts x 3 variants in 44s


,rate
correct_accepted,1.0
corrupted_caught,1.0
foreign_caught,1.0


In [6]:
ex = pd.DataFrame(st["examples"])
print("Misclassified examples:")
ex[((ex.kind == "correct") & ~ex.passed) | ((ex.kind != "correct") & ex.passed)][["kind", "claim", "entailment", "reason"]].head(12)

Misclassified examples:


,kind,claim,entailment,reason
